vamos baixar dados sissmicos de 2016 e 2017

In [2]:
!pip install pandas

   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ----- ---------------------------------- 1.3/9.7 MB 8.4 MB/s eta 0:00:02
   ------------ --------------------------- 3.1/9.7 MB 8.8 MB/s eta 0:00:01
   --------------------- ------------------ 5.2/9.7 MB 9.1 MB/s eta 0:00:01
   ------------------------------ --------- 7.3/9.7 MB 9.2 MB/s eta 0:00:01
   -------------------------------------- - 9.4/9.7 MB 9.5 MB/s eta 0:00:01
   ---------------------------------------- 9.7/9.7 MB 9.5 MB/s  0:00:01

   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   ------------------

In [ ]:
import os, json, time, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

import platform
from pathlib import Path
from shutil import copy2

SO = platform.system()

# ── CAMINHOS ─────────────────────────────────────────────────────
if SO == 'Windows':
    # Pasta de dados (Google Drive)
    PASTA_RAIZ    = r"G:\Meu Drive\TCC\data\scedc-pds"
    # Artefatos do projeto (CSV/logs etc.)
    PASTA_PROJETO = r"C:\Users\vish8\OneDrive\Documentos\GitHub\TCC\Trabalho\artefacts"
else:
    PASTA_RAIZ    = "/Users/alvarosamp/Documents/Projetos/TCC/data/scedc-pds"
    PASTA_PROJETO = "/Users/alvarosamp/Documents/Projetos/TCC/Trabalho/artefacts"

PASTA_WAVE    = os.path.join(PASTA_RAIZ, "event_waveforms")
PASTA_XML     = os.path.join(PASTA_RAIZ, "FDSNstationXML")
PASTA_EXTRA   = os.path.join(PASTA_WAVE, "extra")
INVENTARIO    = os.path.join(PASTA_PROJETO, "data", "inventario_dados.csv")
INVENTARIO_V2 = os.path.join(PASTA_PROJETO, "data", "inventario_dados_v2.csv")
LOG_PATH      = os.path.join(PASTA_PROJETO, "data", "download_log.json")

os.makedirs(PASTA_EXTRA, exist_ok=True)
os.makedirs(os.path.join(PASTA_PROJETO, "data"), exist_ok=True)
os.makedirs(PASTA_XML, exist_ok=True)

# ── PARÂMETROS DE DOWNLOAD ────────────────────────────────────────
LIMITE_GB    = 13.0    # para antes de atingir 13 GB (margem de segurança)
MAG_MIN      = 2.0     # magnitude mínima — eventos com boa SNR
MAG_MAX      = 5.5     # magnitude máxima — evita eventos destrutivos
PROF_MAX_KM  = 30      # profundidade máxima — eventos rasos são mais energéticos
PRE_EVENTO   = 15      # segundos de ruído antes do evento
POS_EVENTO   = 60      # segundos de sinal após o evento
DELAY_API    = 0.3     # segundos entre requisições (respeita rate limit)

# Estações para baixar — todas têm XML na sua pasta
ESTACOES = [
    ("AZ", "BZN"),
    ("AZ", "KNW"),
    ("AZ", "RDM"),
    ("AZ", "PFO"),
    ("AZ", "FRD"),
    ("AZ", "CRY"),
    ("AZ", "SMER"),
    ("AZ", "SND"),
    ("AZ", "TRO"),
    ("AZ", "WMC"),
]

# Períodos de busca — 2016 completo + 2017 parcial
PERIODOS = [
    ("2016-01-01", "2016-01-31"),
    ("2016-02-01", "2016-02-29"),
    ("2016-03-01", "2016-03-31"),
    ("2016-04-01", "2016-04-30"),
    ("2016-05-01", "2016-05-31"),
    ("2016-06-01", "2016-06-30"),
    ("2016-07-01", "2016-07-31"),
    ("2016-08-01", "2016-08-31"),
    ("2016-09-01", "2016-09-30"),
    ("2016-10-01", "2016-10-31"),
    ("2016-11-01", "2016-11-30"),
    ("2016-12-01", "2016-12-31"),
    ("2017-01-01", "2017-01-31"),
    ("2017-02-01", "2017-02-28"),
    ("2017-03-01", "2017-03-31"),
    ("2017-04-01", "2017-04-30"),
    ("2017-05-01", "2017-05-31"),
    ("2017-06-01", "2017-06-30"),
]

MAX_POR_PERIODO = 60   # máximo de eventos por mês

# ── XML: se já existe no projeto, copia para o G: para centralizar ──
def _achar_pasta_xml_fallback():
    for base in [Path.cwd(), *Path.cwd().parents][:6]:
        cand = base / "data" / "scedc-pds" / "FDSNstationXML"
        if cand.exists() and cand.is_dir():
            return cand
    return None

def _pasta_tem_xml(pasta: str) -> bool:
    try:
        return any(Path(pasta).rglob("*.xml"))
    except Exception:
        return False

def _copiar_xml_do_fallback(xml_fb: Path, pasta_xml_destino: str):
    dest_base = Path(pasta_xml_destino)
    dest_base.mkdir(parents=True, exist_ok=True)
    copiados = 0
    for rede, estacao in ESTACOES:
        src1 = xml_fb / rede / f"{rede}.{estacao}.xml"
        src = src1 if src1.exists() else None
        if src is None:
            pasta_rede = xml_fb / rede
            if pasta_rede.exists():
                for cand in pasta_rede.rglob("*.xml"):
                    nome = cand.name
                    if (rede in nome) and (estacao in nome):
                        src = cand
                        break
        if src is None or (not src.exists()):
            continue
        dest_dir = dest_base / rede
        dest_dir.mkdir(parents=True, exist_ok=True)
        dest = dest_dir / f"{rede}.{estacao}.xml"
        if dest.exists():
            continue
        try:
            copy2(str(src), str(dest))
            copiados += 1
        except Exception:
            pass
    return copiados

PASTA_XML_ORIG = PASTA_XML
if (not os.path.exists(PASTA_XML_ORIG)) or (not _pasta_tem_xml(PASTA_XML_ORIG)):
    xml_fb = _achar_pasta_xml_fallback()
    if xml_fb:
        print(f"⚠️  StationXML não encontrado (ou vazio) em: {PASTA_XML_ORIG}")
        print(f"   Achei StationXML no projeto em: {xml_fb}")
        n = _copiar_xml_do_fallback(xml_fb, PASTA_XML_ORIG)
        print(f"   Copiados para o G:: {n} arquivos")
    else:
        print(f"⚠️  StationXML não encontrado (ou vazio) em: {PASTA_XML_ORIG}")
        print("   E também não encontrei o fallback data/scedc-pds/FDSNstationXML no projeto.")

# ── FUNÇÕES UTILITÁRIAS ───────────────────────────────────────────
def tamanho_pasta_gb(pasta):
    total = 0
    for root, _, files in os.walk(pasta):
        for f in files:
            try:
                total += os.path.getsize(os.path.join(root, f))
            except:
                pass
    return total / (1024**3)

def encontrar_xml(rede, estacao):
    # tenta o caminho padrão primeiro (mais confiável)
    cand = Path(PASTA_XML) / rede / f"{rede}.{estacao}.xml"
    if cand.exists():
        return str(cand)
    # fallback: procura por substring (caso nomes diferentes)
    for root, _, files in os.walk(PASTA_XML):
        for f in files:
            if f.endswith('.xml') and rede in f and estacao in f:
                return os.path.join(root, f)
    return None

# Verificar XMLs disponíveis
xmls_ok = [(r, e) for r, e in ESTACOES if encontrar_xml(r, e)]
print(f"✅ Configuração carregada")
print(f"   Limite espaço  : {LIMITE_GB} GB")
print(f"   Períodos       : {len(PERIODOS)} meses")
print(f"   Estações       : {len(ESTACOES)} ({len(xmls_ok)} com XML)")
print(f"   Mag            : {MAG_MIN}–{MAG_MAX}")
print()
print("Estações com XML disponível:")
for r, e in xmls_ok:
    print(f"  ✅ {r}.{e}")
for r, e in ESTACOES:
    if (r, e) not in xmls_ok:
        print(f"  ❌ {r}.{e}  (sem XML — será ignorada)")


✅ Configuração carregada
   Limite espaço  : 13.0 GB
   Períodos       : 18 meses
   Estações       : 10 (10 com XML)
   Mag            : 2.0–5.5

Estações com XML disponível:
  ✅ AZ.BZN
  ✅ AZ.KNW
  ✅ AZ.RDM
  ✅ AZ.PFO
  ✅ AZ.FRD
  ✅ AZ.CRY
  ✅ AZ.SMER
  ✅ AZ.SND
  ✅ AZ.TRO
  ✅ AZ.WMC


In [7]:
!pip install obspy

   ---------------------------------------- 0.0/14.8 MB ? eta -:--:--
   -- ------------------------------------- 1.0/14.8 MB 25.4 MB/s eta 0:00:01
   -------- ------------------------------- 3.1/14.8 MB 10.3 MB/s eta 0:00:02
   -------- ------------------------------- 3.1/14.8 MB 10.3 MB/s eta 0:00:02
   ------------------ --------------------- 6.8/14.8 MB 9.5 MB/s eta 0:00:01
   ---------------------------- ----------- 10.5/14.8 MB 12.3 MB/s eta 0:00:01
   --------------------------------- ------ 12.6/14.8 MB 12.5 MB/s eta 0:00:01
   ---------------------------------------- 14.8/14.8 MB 11.5 MB/s  0:00:01
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 4.0/4.0 MB 48.1 MB/s  0:00:00

   ---------------------------------------- 0/2 [lxml]
   ---------------------------------------- 0/2 [lxml]
   ---------------------------------------- 0/2 [lxml]
   ---------------------------------------- 0/2 [lxml]
   ------------------

In [8]:
# ── BAIXAR STATIONXML (RESPONSES) PARA AS ESTAÇÕES ──────────────────
# Salva em: G:\Meu Drive\TCC\data\scedc-pds\FDSNstationXML\<REDE>\REDE.ESTACAO.xml

from obspy.clients.fdsn import Client
import time
from pathlib import Path

client_xml = Client("SCEDC")  # pode trocar por "IRIS" se quiser

PASTA_XML_DESTINO = Path(os.path.join(PASTA_RAIZ, "FDSNstationXML"))
PASTA_XML_DESTINO.mkdir(parents=True, exist_ok=True)

erros_xml = []
baixados_xml = 0

for rede, estacao in ESTACOES:
    pasta_rede = PASTA_XML_DESTINO / rede
    pasta_rede.mkdir(parents=True, exist_ok=True)
    out_file = pasta_rede / f"{rede}.{estacao}.xml"

    if out_file.exists():
        print(f"↩️  Já existe: {out_file.name}")
        continue

    try:
        # BH? cobre BHZ/BHN/BHE quando existir; location="*" pega qualquer location code
        inv = client_xml.get_stations(
            network=rede,
            station=estacao,
            location="*",
            channel="BH?",
            level="response",
        )
        inv.write(str(out_file), format="STATIONXML")
        baixados_xml += 1
        print(f"✅ XML salvo: {out_file}")
    except Exception as e:
        erros_xml.append((rede, estacao, str(e)))
        print(f"❌ Falha XML {rede}.{estacao}: {e}")

    time.sleep(DELAY_API)

print()
print(f"StationXML baixados: {baixados_xml}")
print(f"Falhas: {len(erros_xml)}")
if erros_xml:
    print("Exemplos de falhas:")
    for rede, estacao, msg in erros_xml[:5]:
        print(f"  - {rede}.{estacao}: {msg}")

✅ XML salvo: G:\Meu Drive\TCC\data\scedc-pds\FDSNstationXML\AZ\AZ.BZN.xml
✅ XML salvo: G:\Meu Drive\TCC\data\scedc-pds\FDSNstationXML\AZ\AZ.KNW.xml
✅ XML salvo: G:\Meu Drive\TCC\data\scedc-pds\FDSNstationXML\AZ\AZ.RDM.xml
✅ XML salvo: G:\Meu Drive\TCC\data\scedc-pds\FDSNstationXML\AZ\AZ.PFO.xml
✅ XML salvo: G:\Meu Drive\TCC\data\scedc-pds\FDSNstationXML\AZ\AZ.FRD.xml
✅ XML salvo: G:\Meu Drive\TCC\data\scedc-pds\FDSNstationXML\AZ\AZ.CRY.xml
✅ XML salvo: G:\Meu Drive\TCC\data\scedc-pds\FDSNstationXML\AZ\AZ.SMER.xml
✅ XML salvo: G:\Meu Drive\TCC\data\scedc-pds\FDSNstationXML\AZ\AZ.SND.xml
✅ XML salvo: G:\Meu Drive\TCC\data\scedc-pds\FDSNstationXML\AZ\AZ.TRO.xml
✅ XML salvo: G:\Meu Drive\TCC\data\scedc-pds\FDSNstationXML\AZ\AZ.WMC.xml

StationXML baixados: 10
Falhas: 0


In [10]:
from obspy.clients.fdsn import Client
from obspy import UTCDateTime, Stream

client = Client("SCEDC")

# Carregar log de downloads anteriores (para retomar se interrompido)
if os.path.exists(LOG_PATH):
    with open(LOG_PATH) as f:
        log = json.load(f)
    arquivos_ja_baixados = set(log.get('baixados', []))
    print(f"Retomando download anterior: {len(arquivos_ja_baixados)} já baixados")
else:
    log = {'baixados': [], 'erros': [], 'iniciado': str(pd.Timestamp.now())}
    arquivos_ja_baixados = set()

# Mapear arquivos já existentes em disco
for root, _, files in os.walk(PASTA_WAVE):
    for f in files:
        if f.endswith('.ms'):
            arquivos_ja_baixados.add(f)

print(f"Arquivos em disco: {len(arquivos_ja_baixados)}")
print(f"Espaço usado agora: {tamanho_pasta_gb(PASTA_WAVE):.2f} GB")
print()

novos_arquivos = []
total_baixados = 0
total_erros    = 0
t_inicio       = time.time()

for idx_periodo, (p_ini, p_fim) in enumerate(PERIODOS):
    
    # Verificar espaço antes de cada período
    gb_usado = tamanho_pasta_gb(PASTA_WAVE)
    if gb_usado >= LIMITE_GB:
        print(f"\n🛑 Limite de {LIMITE_GB} GB atingido ({gb_usado:.2f} GB). Parando.")
        break
    
    print(f"\n[{idx_periodo+1}/{len(PERIODOS)}] {p_ini} → {p_fim}  "
          f"({gb_usado:.2f}/{LIMITE_GB} GB)")
    
    # Buscar catálogo
    try:
        cat = client.get_events(
            starttime    = UTCDateTime(p_ini),
            endtime      = UTCDateTime(p_fim),
            minmagnitude = MAG_MIN,
            maxmagnitude = MAG_MAX,
            maxdepth     = PROF_MAX_KM * 1000,
            limit        = MAX_POR_PERIODO,
            orderby      = 'mag'  # SCEDC aceita: time, time-asc, mag, mag-asc
        )
    except Exception as e:
        print(f"  ⚠️  Erro no catálogo: {e}")
        total_erros += 1
        continue
    
    print(f"  {len(cat)} eventos encontrados")
    baixados_mes = 0
    
    for evento in cat:
        
        # Checar espaço a cada evento
        if tamanho_pasta_gb(PASTA_WAVE) >= LIMITE_GB:
            print(f"  🛑 Limite atingido durante o período. Parando.")
            break
        
        try:
            origem = evento.origins[0]
            mag    = evento.magnitudes[0].mag if evento.magnitudes else 0
            t0     = origem.time
            
            nome = f"extra_{t0.strftime('%Y%m%d_%H%M%S')}_M{mag:.1f}.ms"
            
            if nome in arquivos_ja_baixados:
                continue
            
            # Baixar todas as estações de uma vez
            st_final = Stream()
            for rede, estacao in xmls_ok:
                try:
                    st = client.get_waveforms(
                        network   = rede,
                        station   = estacao,
                        location  = "*",
                        channel   = "BHZ",
                        starttime = t0 - PRE_EVENTO,
                        endtime   = t0 + POS_EVENTO
                    )
                    st_final += st
                    time.sleep(DELAY_API)
                except:
                    pass
            
            if len(st_final) == 0:
                continue
            
            # Salvar
            caminho = os.path.join(PASTA_EXTRA, nome)
            st_final.write(caminho, format="MSEED")
            
            arquivos_ja_baixados.add(nome)
            novos_arquivos.append({
                'arquivo'  : nome,
                'caminho'  : caminho,
                'origem'   : str(t0),
                'mag'      : float(mag),
                'n_traces' : len(st_final),
                'gb_acum'  : tamanho_pasta_gb(PASTA_WAVE),
            })
            
            baixados_mes  += 1
            total_baixados += 1
            
            # Salvar log periodicamente
            log['baixados'] = list(arquivos_ja_baixados)
            with open(LOG_PATH, 'w') as f:
                json.dump(log, f)
            
            elapsed = time.time() - t_inicio
            gb_now  = tamanho_pasta_gb(PASTA_WAVE)
            print(f"  ✅ {nome}  "
                  f"M{mag:.1f}  "
                  f"traces={len(st_final)}  "
                  f"{gb_now:.2f}GB  "
                  f"({elapsed/60:.0f}min)")
            
        except Exception as e:
            total_erros += 1
            continue
    
    print(f"  → {baixados_mes} novos neste período")

# Resumo final
elapsed_total = time.time() - t_inicio
gb_final      = tamanho_pasta_gb(PASTA_WAVE)

print()
print("=" * 65)
print(f"DOWNLOAD CONCLUÍDO")
print(f"  Novos arquivos  : {total_baixados}")
print(f"  Erros ignorados : {total_erros}")
print(f"  Espaço usado    : {gb_final:.2f} GB")
print(f"  Tempo total     : {elapsed_total/60:.0f} minutos")


Arquivos em disco: 0
Espaço usado agora: 0.00 GB


[1/18] 2016-01-01 → 2016-01-31  (0.00/13.0 GB)
  60 eventos encontrados
  ✅ extra_20160106_131855_M5.3.ms  M5.3  traces=1  0.00GB  (0min)
  ✅ extra_20160130_002841_M5.1.ms  M5.1  traces=1  0.00GB  (0min)
  ✅ extra_20160130_002841_M5.0.ms  M5.0  traces=1  0.00GB  (0min)
  ✅ extra_20160130_003052_M5.0.ms  M5.0  traces=1  0.00GB  (0min)
  ✅ extra_20160102_051146_M4.4.ms  M4.4  traces=1  0.00GB  (0min)
  ✅ extra_20160106_144234_M4.4.ms  M4.4  traces=1  0.00GB  (0min)
  ✅ extra_20160107_054952_M4.3.ms  M4.3  traces=1  0.00GB  (0min)
  ✅ extra_20160128_092214_M4.3.ms  M4.3  traces=1  0.00GB  (0min)
  ✅ extra_20160115_223726_M4.3.ms  M4.3  traces=1  0.00GB  (0min)
  ✅ extra_20160124_153216_M4.1.ms  M4.1  traces=1  0.00GB  (0min)
  ✅ extra_20160129_102131_M4.1.ms  M4.1  traces=1  0.00GB  (0min)
  ✅ extra_20160129_093509_M4.1.ms  M4.1  traces=1  0.00GB  (0min)
  ✅ extra_20160119_102106_M3.6.ms  M3.6  traces=1  0.00GB  (0min)
  ✅ extra_20160104_